# Ethics Course Assistant - RAG Comparison

Comparison of RAG, TF-IDF and LLM for course Q&A.


In [11]:
# Setup: auto-reload modules during development
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
from dotenv import load_dotenv
load_dotenv()
print("API Key:", "OK" if os.getenv("GEMINI_API_KEY") else "Missing")


API Key: OK


## Load Documents


In [13]:
from corpus import load_documents
docs = load_documents("sources")
print(f"Loaded {len(docs)} chunks")


Loaded 55 chunks


## Search Engines


In [14]:
from retrieval import KeywordSearch, SemanticSearch, LLM

kw_search = KeywordSearch(docs)
sem_search = SemanticSearch(docs)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

## QA Systems


In [15]:
from retrieval import RAG, DirectLLM

llm = LLM()

rag_sys = RAG(sem_search, llm, k=5, threshold=0.35)
tfidf_sys = RAG(kw_search, llm, k=5, threshold=0.08)
llm_sys = DirectLLM(llm)


## Test Query


In [ ]:
q = "What is the Euthyphro dilemma?"

print("RAG:", rag_sys.ask(q).text[:200])
print("\nTF-IDF:", tfidf_sys.ask(q).text[:200])
print("\nLLM:", llm_sys.ask(q).text[:200])


## Retrieval Metrics


In [ ]:
from retrieval import calc_recall, calc_mrr
from corpus import QUESTIONS

print("Semantic - Recall:", calc_recall(sem_search, QUESTIONS, 5, 0.35))
print("Semantic - MRR:", calc_mrr(sem_search, QUESTIONS, 5, 0.35))
print("TF-IDF - Recall:", calc_recall(kw_search, QUESTIONS, 5, 0.08))
print("TF-IDF - MRR:", calc_mrr(kw_search, QUESTIONS, 5, 0.08))


Semantic - Recall: 0.7142857142857143
Semantic - MRR: 0.4571428571428572
TF-IDF - Recall: 0.5714285714285714
TF-IDF - MRR: 0.5


## Evaluation


In [ ]:
from corpus import evaluate, show_results

print("RAG:")
s1 = evaluate(rag_sys, docs)

print("\nTF-IDF:")
s2 = evaluate(tfidf_sys, docs)

print("\nLLM Only:")
s3 = evaluate(llm_sys, docs)

show_results(s1, s2, s3)


RAG:
Questions with answers:
  [1.0] What philosophical puzzle asks whether holine... -> 095d54aa351c9a2e1a79d2e8a1ce0201_MIT24_231F09_lec02.pdf, p.1
  [1.0] What ethical theory claims that God's will de... -> 095d54aa351c9a2e1a79d2e8a1ce0201_MIT24_231F09_lec02.pdf, p.2
  [1.0] How did the British philosopher argue that go... -> 4d3b71ffcc7a256a2dcee44f97277875_MIT24_231F09_lec03.pdf, p.3
  [1.0] What is the difference between reasons that a... -> 53a0477659ae023edf3de9c60975f622_MIT24_231F09_lec15.pdf, p.2
  [1.0] According to the course, what types of person... -> 2aa5e59ac0dd208cb8d8d58c872aa228_MIT24_231F09_lec18.pdf, p.4
  [0.2] Why might following the principle of maximizi... -> 6dcdb0d5f044a3dc395fb57acb948430_MIT24_231F09_lec13.pdf, p.3
  [0.2] Why can't we know if our actions are truly ri... -> 53a0477659ae023edf3de9c60975f622_MIT24_231F09_lec15.pdf, p.2
Unanswerable:
  [1.0] What percentage of students passed the Ethics 24.2...
  [1.0] How many office hours per week did the p

In [ ]:
from corpus import QUESTIONS, UNANSWERABLE, OFFTOPIC

print("=== RAG Answers ===")
for q in QUESTIONS:
    r = rag_sys.ask(q.text)
    print(f"\nQ: {q.text}")
    print(f"A: {r.text}")

print("\n=== Unanswerable ===")
for q in UNANSWERABLE:
    r = rag_sys.ask(q)
    print(f"\nQ: {q}")
    print(f"A: {r.text}")

print("\n=== Off-topic ===")
for q in OFFTOPIC:
    r = rag_sys.ask(q)
    print(f"\nQ: {q}")
    print(f"A: {r.text}")

## Interactive


In [ ]:
question = "What does Moore's Open Question Argument prove?"
r = rag_sys.ask(question)
print("Answer:", r.text)
print("Sources:", len(r.refs))


Answer: Moore’s Open Question Argument proves that good couldn’t be identical to any natural property, because any such identity claim seems to invite an “open question.” Specifically, it shows that if a question is open – meaning we can ask “Is X Y?” without seeming incompetent with the words – then “X” does not have the same meaning as “Y,” and therefore the things they refer to are not identical.




Sources: 5


## 8. Search Comparison Demo

See how semantic vs keyword search differs for the same query.


In [ ]:
q = "Why might consequences be unknowable?"

print("Semantic:")
for d in sem_search.find(q, 3, 0.3):
    print(" ", d.split('\n')[0])

print("\nTF-IDF:")
for d in kw_search.find(q, 3, 0.05):
    print(" ", d.split('\n')[0])


Semantic:
  [Source: 53a0477659ae023edf3de9c60975f622_MIT24_231F09_lec15.pdf | Page 2]
  [Source: 6dcdb0d5f044a3dc395fb57acb948430_MIT24_231F09_lec13.pdf | Page 3]
  [Source: 6dcdb0d5f044a3dc395fb57acb948430_MIT24_231F09_lec13.pdf | Page 5]

TF-IDF:
  [Source: 5a1e538bccd9add8d5abad4c21fc4a1c_MIT24_231F09_lec16.pdf | Page 1]
  [Source: 53a0477659ae023edf3de9c60975f622_MIT24_231F09_lec15.pdf | Page 2]
  [Source: 53a0477659ae023edf3de9c60975f622_MIT24_231F09_lec15.pdf | Page 1]
